<b>Group Number:</b>
<br><b>Name Group Member 1: Anton Seifert</b>
<br><b>u-Kürzel Group Member 1: unjud</b>
<br><b>Name Group Member 2: Arne Segatz</b>
<br><b>u-Kürzel Group Member 2: urnye</b>

# Unit 4: Data Splitting for Evaluation

## Introduction

In the preceding units, we undertook the essential tasks of loading, exploring, cleaning, and preprocessing the Bank Marketing dataset. This culminated in a processed feature matrix (`X_processed`) and a corresponding target vector (`y_processed`), theoretically ready for input into machine learning algorithms.

However, a critical step must occur *before* any model training begins: **splitting the data**. Why is this non-negotiable? If we train a model and evaluate its performance on the *exact same data* it learned from, the results will almost certainly be overly optimistic. The model might simply memorise the training examples, including their noise and specific idiosyncrasies (a phenomenon called **overfitting**), rather than learning the underlying generalisable patterns. Consequently, it would likely perform poorly when faced with new, previously unseen data.

Our true goal is to build models that **generalise** well. To assess this generalisation ability reliably, we must evaluate the model on data it has never encountered during training. This unit covers the standard techniques for achieving this separation:

1.  **Train/Test Split:** The fundamental division of data into a training set (for model learning) and a test set (for final, unbiased evaluation).
2.  **Stratified Sampling:** A crucial refinement for classification tasks, ensuring that the proportion of different classes in the original dataset is preserved in both the training and test splits. This is particularly important for imbalanced datasets like ours.
3.  **k-Fold Cross-Validation:** A more robust technique for estimating model performance and for tasks like hyperparameter tuning, performed by creating multiple train/validation splits *within* the main training dataset.

In [20]:
import numpy as np
import os
from sklearn.model_selection import train_test_split, StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Processed Data (Saved from Unit 2) ---
X_processed = None
y_processed = None
data_dir = "data"
x_load_path = os.path.join(data_dir, "bank_X_processed.npy")
y_load_path = os.path.join(data_dir, "bank_y_processed.npy")

X_processed = np.load(x_load_path, allow_pickle=True)
y_processed = np.load(y_load_path, allow_pickle=True)
print("Successfully loaded processed data (X_processed, y_processed).")
print(f"X_processed shape: {X_processed.shape}")
print(f"y_processed shape: {y_processed.shape}")

Successfully loaded processed data (X_processed, y_processed).
X_processed shape: (4521, 37)
y_processed shape: (4521,)


## 4.1 The Train/Test Split

he most common and fundamental way to prepare data for evaluation is the **train/test split**. We partition the entire processed dataset into two distinct, non-overlapping sets:

*   **Training Set:** This subset (typically 70-80% of the data) is used exclusively by the machine learning algorithm to learn patterns, relationships, and internal parameters. The model "sees" this data during the fitting process.
*   **Test Set (or Hold-out Set):** This subset (the remaining 20-30%) is kept completely separate and is used **only once** at the very end of the modelling process. Its sole purpose is to provide an unbiased evaluation of the final, trained model's performance on data it has never encountered before.

**Why is the separation so strict?**
Using the test set for *any* part of the model development process other than the final evaluation can lead to **data leakage**. This means information from the test set inadvertently influences the model's training or selection, causing the reported test performance to be artificially inflated and not reflective of true generalisation ability. Activities like feature selection, model selection based on test scores, or hyperparameter tuning using the test set must be avoided.

**Key Parameters for Splitting:**

*   **`test_size` (or `train_size`):** Specifies the proportion (or absolute number) of data points to allocate to the test (or train) set.
*   **`random_state`:** A pseudo-random number generator seed. Setting this to a fixed integer ensures that the *same* random split is generated every time the code runs. This is crucial for **reproducibility** – allowing you or others to get the exact same results – and for fair comparison between different modelling experiments.
*   **`shuffle`:** Whether to shuffle the data *before* splitting. This is generally recommended (`True` by default) to ensure that any inherent ordering in the original dataset doesn't bias the split (e.g., preventing all the 'yes' cases from ending up in one set if they happened to be grouped together).
*   **`stratify`:** This is essential for classification. When set to the target variable (`y`), it ensures that the proportion of each class in the output splits matches the proportion in the input data. For our imbalanced Bank Marketing dataset, using `stratify=y_processed` is vital to prevent the test set (or training set) from having a wildly different percentage of 'yes'/'no' cases than the overall dataset.

<div class="alert alert-block alert-info">
    <b>Further Reading:</b>
    <ul>
        <li><a href="https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html" target="_blank">Scikit-learn: `train_test_split` Documentation</a></li>
        <li><a href="https://machinelearningmastery.com/data-leakage-machine-learning/" target="_blank">Machine Learning Mastery: Data Leakage in Machine Learning</a></li>
    </ul>
</div>

<div class="alert alert-block alert-success">
<b>Task 4.1: Stratified Train/Test Split (2 pts)</b>

<ul>
    <li> Use <code>sklearn.model_selection.train_test_split</code> to split <i>X_processed</i> and <i>y_processed</i> into training and testing sets.</li>
    <li> Assign the results to variables <i>X_train</i>, <i>X_test</i>, <i>y_train</i>, <i>y_test</i>.</li>
    <li> Print the shapes of the resulting arrays.</li>
    <li> Calculate and print the proportion of the positive class (y=1) in the original set, the training set, and the test set to verify stratification.</li>
</ul>
</div>

In [21]:
print("--- Performing Stratified Train/Test Split ---")
test_proportion = 0.25
seed = 42

X_train: np.array = None
y_train: np.array = None

X_test: np.array = None
y_test: np.array = None

### STUDENT CODE HERE (2 pts)
X_train, X_test, y_train, y_test = train_test_split(X_processed, y_processed, test_size=test_proportion, random_state= seed, stratify=y_processed)#

#print shapes of you by ed sheeren
print("shapes:")
print("Xtrain: {}, Ytrain: {}, XTest: {}, YTest: {}".format(X_train.shape, y_train.shape, X_test.shape, y_test.shape))

#verifying stratification
print("verifying stratification:")
print(" Yprocessed: {}, Ytrain: {},  YTest: {}".format(y_processed.mean(), y_train.mean(),  y_test.mean()))

### STUDENT CODE until HERE

--- Performing Stratified Train/Test Split ---
shapes:
Xtrain: (3390, 37), Ytrain: (3390,), XTest: (1131, 37), YTest: (1131,)
verifying stratification:
 Yprocessed: 0.11523999115239991, Ytrain: 0.11533923303834809,  YTest: 0.11494252873563218


## 4.2 k-Fold Cross-Validation (CV)

While the train/test split provides a necessary final evaluation set, relying on just one split to *compare* different models or *tune* a model's hyperparameters can be unreliable. The performance score obtained on that single test set might be overly optimistic or pessimistic purely due to the specific random sample of data points that landed in it.

**k-Fold Cross-Validation (CV)** offers a more robust approach for these intermediate evaluation tasks (model selection, hyperparameter tuning). It systematically creates and evaluates on multiple train/validation splits *within the main training dataset*, thereby reducing the variance associated with a single split.

**The k-Fold CV Process:**

1.  **Partition Training Data:** The **training set** (`X_train`, `y_train` from the initial split) is divided into *k* roughly equal-sized, non-overlapping subsets called "folds" (common values for *k* are 5 or 10).
2.  **Iterative Training & Validation:** The process iterates *k* times. In each iteration `i`:
    *   **Validation Fold:** Fold `i` is designated as the temporary validation set for this iteration.
    *   **Training Folds:** The remaining `k-1` folds are combined to form the training set for this iteration.
    *   **Model Fitting & Evaluation:** The chosen model (or model configuration) is trained on the `k-1` training folds and then evaluated on the validation fold `i`. The performance score (e.g., accuracy, AUC, F1-score) is recorded.
3.  **Aggregate Results:** After all *k* iterations are complete, we have *k* individual performance scores. The **average** of these scores serves as the overall cross-validation performance estimate for the model (or configuration). The **standard deviation** of these scores provides insight into the stability or variance of the model's performance across different subsets of the training data.

**Stratified k-Fold for Classification:**
Just as stratification was important for the initial train/test split in our imbalanced dataset, it's equally important within cross-validation. **Stratified k-Fold CV** (`StratifiedKFold` in Scikit-learn) ensures that each of the *k* folds maintains approximately the same percentage of samples for each target class as found in the complete training set. This prevents folds from being heavily skewed towards one class, leading to more reliable evaluation, especially for metrics sensitive to class balance like F1-score or recall on the minority class.

**Key Usage:**
*   Use CV **only on the training data** (`X_train`, `y_train`).
*   Use the average CV score to compare different algorithms or different hyperparameter settings.
*   The final test set (`X_test`, `y_test`) remains untouched during CV.

<div class="alert alert-block alert-info">
    <b>Further Reading:</b>
    <ul>
        <li><a href="https://scikit-learn.org/stable/modules/cross_validation.html" target="_blank">Scikit-learn: Cross-validation User Guide</a></li>
        <li><a href="https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html" target="_blank">Scikit-learn: `StratifiedKFold` Documentation</a></li>
        <li><a href="https://machinelearningmastery.com/k-fold-cross-validation/" target="_blank">Machine Learning Mastery: A Gentle Introduction to k-fold Cross-Validation</a></li>
    </ul>
</div>

<p align='center'>
    <img src ="https://scikit-learn.org/stable/_images/grid_search_cross_validation.png" width=60% height = "auto"/>
    <br>
    <em> Figure 1: Visualisation of K-Fold. </em>
</p>

<div class="alert alert-block alert-success">
<b>Task 4.2: Demonstrating Stratified K-Fold CV (1.5 pts)</b>

<ul>
    <li> Instantiate <code>sklearn.model_selection.StratifiedKFold</code> with the provided parameters.</li>
    <li> Use the <code>.split()</code> method of the `StratifiedKFold` object on the **training data** (`X_train`, `y_train`) to generate indices for each fold.</li>
    <li> For each fold, print:
        <ul>
            <li>The fold number (1 to 5).</li>
            <li>The number of samples in the training part for that fold.</li>
            <li>The number of samples in the validation part for that fold.</li>
            <li>The proportion of the positive class (y=1) within that fold's <b>validation set</b>.</li>
        </ul>
    </li>
    <li> Compare the validation proportions to the overall training set proportion.</li>
</ul>
</div>

In [22]:
print("\n--- Demonstrating 5-Fold Stratified Cross-Validation ---")
n_splits_cv = 5
seed_cv = 42

### STUDENT CODE HERE (1.5 pts) 

skf = StratifiedKFold(n_splits= n_splits_cv, shuffle= True, random_state= seed_cv)
for i, (train_set, test_set) in enumerate(skf.split(X_train, y_train)):#split() returns indexes of splitted set
    print("fold", i)
    print("samples in train:{}, samples in test: {}".format(len(train_set), len(test_set)))
    print("proportion pos", y_train[test_set].mean(), "\n")
    



### STUDENT CODE until HERE 

print(f"\nOverall training set Class 1 proportion: {np.mean(y_train):.4f}")


--- Demonstrating 5-Fold Stratified Cross-Validation ---
fold 0
samples in train:2712, samples in test: 678
proportion pos 0.11504424778761062 

fold 1
samples in train:2712, samples in test: 678
proportion pos 0.11504424778761062 

fold 2
samples in train:2712, samples in test: 678
proportion pos 0.11504424778761062 

fold 3
samples in train:2712, samples in test: 678
proportion pos 0.11504424778761062 

fold 4
samples in train:2712, samples in test: 678
proportion pos 0.11651917404129794 


Overall training set Class 1 proportion: 0.1153


## 4.3 Saving the Splits

We have successfully created our primary stratified train/test split (`X_train`, `X_test`, `y_train`, `y_test`). To ensure we use precisely these same splits in the subsequent units where we will train and evaluate models, it is crucial to save these arrays to disk. This avoids variations caused by rerunning the `train_test_split` function (even with the same `random_state`, it's good practice to save definitive splits).

We will use NumPy's native binary file format (`.npy`) for efficient storage and retrieval.

<div class="alert alert-block alert-success">
<b>Task 4.3: Saving Train/Test Splits (0.75 pt)</b>

<ul>
    <li> Save the files to the provided paths.</li>
</ul>
</div>

In [23]:
print("\n--- Saving Train/Test Splits ---")
data_dir = "data"

xtrain_save_path = os.path.join(data_dir, "bank_X_train_strat.npy")
xtest_save_path = os.path.join(data_dir, "bank_X_test_strat.npy")
ytrain_save_path = os.path.join(data_dir, "bank_y_train_strat.npy")
ytest_save_path = os.path.join(data_dir, "bank_y_test_strat.npy")

### STUDENT CODE HERE (0.75 pt) 
np.save(xtrain_save_path, X_train)
np.save(xtest_save_path, X_test)
np.save(ytrain_save_path, y_train)
np.save(ytest_save_path, y_test)


### STUDENT CODE until HERE 


--- Saving Train/Test Splits ---


## Unit 4 Summary & Deep Dive Topics

In this unit, we addressed the critical step of splitting our processed data for reliable model evaluation. We learned about and implemented:

*   **Train/Test Split:** The fundamental practice of separating data into a training set for model learning and a test set for unbiased final evaluation, crucial for assessing generalisation.
*   **Stratification:** The technique of ensuring that the class proportions within the train and test splits accurately reflect the original dataset's distribution. We saw why this is essential for our imbalanced Bank Marketing dataset using the `stratify` parameter.
*   **k-Fold Cross-Validation (Stratified):** A robust method for estimating model performance and performing model selection/tuning by creating multiple train/validation splits *within* the main training set, again using stratification (`StratifiedKFold`) to handle class imbalance appropriately.

We concluded by saving our definitive stratified train/test splits (`X_train`, `X_test`, `y_train`, `y_test`) to `.npy` files. This ensures that these exact splits can be consistently loaded and used in the subsequent units focused on model training and detailed evaluation, promoting reproducibility and reliable comparisons.

### Deep Dive Topics

#### 1. Leave-One-Out Cross-Validation (LOOCV):

LOOCV is a specific, exhaustive type of k-fold cross-validation where the number of folds (`k`) is equal to the number of samples (`n`) in the dataset.

*   **Process:** In each iteration, one single data point is held out as the validation set, and the model is trained on the remaining `n-1` data points. This process is repeated `n` times, with each data point serving as the validation set exactly once. The final performance metric is typically the average of the `n` individual validation scores.
*   **Relationship to k-Fold:** It's the most extreme version of k-fold CV.
*   **Pros:**
    *   **Low Bias:** Because almost the entire dataset (`n-1` samples) is used for training in each iteration, the performance estimate tends to have low bias. It closely reflects the performance of a model trained on the full dataset.
    *   **Uses Data Efficiently:** It's particularly appealing when you have a **very small dataset**, as it maximizes the amount of data used for training in each fold. There's no "wasted" data sitting out in larger validation folds.
    *   **Deterministic:** There's no randomness in the split itself (unlike shuffled k-fold), so the result is always the same for a given dataset.
*   **Cons:**
    *   **Computationally Very Expensive:** Requires training the model `n` times. This can be prohibitive for large datasets or complex models that take a long time to train.
    *   **High Variance:** The performance estimate derived from LOOCV can have high variance. This is somewhat counter-intuitive. Because the training sets in each fold are extremely similar (differing by only one sample swap), the resulting models are highly correlated. The average of these highly correlated model evaluations can fluctuate significantly if the dataset changes slightly. In contrast, k-fold CV (with smaller k, like 5 or 10) trains models on more diverse subsets, leading to less correlated evaluations and often a more stable (lower variance) final estimate, even if slightly more biased.
*   **When to Consider:** Primarily for **very small datasets** where maximizing training data in each fold is critical and the computational cost is manageable. For larger datasets, k-fold CV (k=5 or 10) usually offers a better balance between bias, variance, and computational cost.
    *   *Check out:* `sklearn.model_selection.LeaveOneOut` in Scikit-learn.

<p align='center'>
    <img src="https://dataaspirant.com/wp-content/uploads/2023/10/1-3-1536x933.png" width=60% height = "auto"/>
    <em> Figure 2: Leave one out cross validation. </em>
</p>

*   **Further Reading & Resources:**
    *   Scikit-learn Documentation: [`sklearn.model_selection.LeaveOneOut`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneOut.html)
    *   Machine Learning Mastery: [LOOCV for Evaluating Machine Learning Algorithms](https://machinelearningmastery.com/loocv-for-evaluating-machine-learning-algorithms/)

#### 2. Impact of `random_state`:

Many operations in machine learning involve randomness, including splitting data (`train_test_split` with shuffling) and initializing cross-validation folds (`KFold`, `StratifiedKFold` when `shuffle=True`). The `random_state` parameter, typically accepting an integer value, acts as a **seed** for the pseudo-random number generator used in these operations.

*   **Purpose: Reproducibility:** By setting `random_state` to a specific integer (e.g., `random_state=42`), you ensure that the sequence of "random" numbers generated for shuffling or splitting will be **exactly the same** every time you run the code. This means:
    *   Your train/test split will always contain the same data points.
    *   Your cross-validation folds will always be generated in the same way (if `shuffle=True`).
*   **Why is Reproducibility Important?**
    *   **Debugging:** If you encounter an issue, you can reliably reproduce the exact conditions that caused it.
    *   **Comparison:** When comparing different models, feature sets, or hyperparameters, using the same `random_state` ensures that any observed performance differences are due to the changes you made, not due to random variations in the data splits.
    *   **Collaboration & Reporting:** Allows others (or your future self) to reproduce your results exactly.
*   **What Happens if `random_state` is Omitted (or `None`)?**
    *   If `random_state` is not set, the random number generator is typically initialized using a system-dependent source (like the current time or OS-specific sources).
    *   This means that **every time you run the code, you will get a different random split** or different shuffling for CV folds.
    *   While each individual run might be valid, it makes debugging extremely difficult and comparing results across different runs unreliable. Small changes in performance could just be noise from the different splits.
*   **Best Practice:** Always set `random_state` to a fixed integer during development, experimentation, and reporting for any operation involving randomness (like splitting, CV shuffling, some model initializations, etc.). The specific number (e.g., 0, 42, 123) doesn't matter, only that it's consistent.

*   **Further Reading & Resources:**
    *   Scikit-learn Documentation: [Controlling randomness in scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html#controlling-randomness) or [Glossary entry for `random_state`](https://scikit-learn.org/stable/glossary.html#term-random_state)
    *   Stack Overflow Discussion: [What does random_state do?](https://stackoverflow.com/questions/28064634/random-state-pseudo-random-number-in-scikit-learn)
    *   Medium Article: [Why do we set a random state in machine learning models?](https://medium.com/data-science/why-do-we-set-a-random-state-in-machine-learning-models-bb2dc68d8431) 